In [1]:
import pandas as pd
import numpy as np

import time
import datetime as dt

from scipy.stats import norm

In [2]:
csv_options = pd.read_csv('icecleared_gasoptions_2026_03_17.csv', header=0, index_col=0)
csv_underlying = pd.read_csv("icecleared_gas_2026_03_17.csv", header=0, index_col = 0)

In [3]:
index = "JKM"
df1_underlying = csv_underlying[csv_underlying.CONTRACT == "JKM"]
df1_opt = csv_options[csv_options.CONTRACT == "JKM"]

In [4]:
df = pd.merge(df1_opt, df1_underlying[["SETTLEMENT PRICE", "EXPIRATION DATE"]], on ="EXPIRATION DATE", how="inner")

In [5]:
print(df1_opt.shape)
print(df.shape)

(600, 12)
(600, 13)


In [6]:
df

,HUB,PRODUCT,STRIP,CONTRACT,CONTRACT TYPE,STRIKE,SETTLEMENT PRICE_x,NET CHANGE,EXPIRATION DATE,PRODUCT_ID,OPTION_VOLATILITY,DELTA_FACTOR,SETTLEMENT PRICE_y
0,JKM,LNG Futures,5/1/2026,JKM,C,5.0,14.414,-0.119,4/15/2026,4172,103.7693,1.0,19.414
1,JKM,LNG Futures,5/1/2026,JKM,C,8.0,11.414,-0.119,4/15/2026,4172,68.3377,1.0,19.414
2,JKM,LNG Futures,5/1/2026,JKM,C,9.0,10.414,-0.119,4/15/2026,4172,59.4640,1.0,19.414
3,JKM,LNG Futures,5/1/2026,JKM,C,10.0,9.414,-0.119,4/15/2026,4172,51.5178,1.0,19.414
4,JKM,LNG Futures,5/1/2026,JKM,C,11.5,7.914,-0.119,4/15/2026,4172,40.9523,1.0,19.414
...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,JKM,LNG Futures,6/1/2033,JKM,F,NaN,9.176,0.193,5/13/2033,4172,NaN,NaN,9.176
596,JKM,LNG Futures,7/1/2033,JKM,F,NaN,9.084,0.193,6/15/2033,4172,NaN,NaN,9.084
597,JKM,LNG Futures,8/1/2033,JKM,F,NaN,9.256,0.194,7/15/2033,4172,NaN,NaN,9.256
598,JKM,LNG Futures,9/1/2033,JKM,F,NaN,9.407,0.194,8/15/2033,4172,NaN,NaN,9.407


In [7]:
today = dt.date.today()
today

datetime.date(2026, 3, 20)

In [8]:
df["EXPIRATION DATE"] = pd.to_datetime(df["EXPIRATION DATE"], format="%m/%d/%Y").dt.date
df["EXPIRATION DATE"]

0      2026-04-15
1      2026-04-15
2      2026-04-15
3      2026-04-15
4      2026-04-15
          ...    
595    2033-05-13
596    2033-06-15
597    2033-07-15
598    2033-08-15
599    2033-09-15
Name: EXPIRATION DATE, Length: 600, dtype: object

In [9]:
df["TIME TO MATURITY"] = (df["EXPIRATION DATE"] - today).dt.days
df["TIME TO MATURITY-365"] = df["TIME TO MATURITY"]/365

In [10]:
df

,HUB,PRODUCT,STRIP,CONTRACT,CONTRACT TYPE,STRIKE,SETTLEMENT PRICE_x,NET CHANGE,EXPIRATION DATE,PRODUCT_ID,OPTION_VOLATILITY,DELTA_FACTOR,SETTLEMENT PRICE_y,TIME TO MATURITY,TIME TO MATURITY-365
0,JKM,LNG Futures,5/1/2026,JKM,C,5.0,14.414,-0.119,2026-04-15,4172,103.7693,1.0,19.414,26,0.071233
1,JKM,LNG Futures,5/1/2026,JKM,C,8.0,11.414,-0.119,2026-04-15,4172,68.3377,1.0,19.414,26,0.071233
2,JKM,LNG Futures,5/1/2026,JKM,C,9.0,10.414,-0.119,2026-04-15,4172,59.4640,1.0,19.414,26,0.071233
3,JKM,LNG Futures,5/1/2026,JKM,C,10.0,9.414,-0.119,2026-04-15,4172,51.5178,1.0,19.414,26,0.071233
4,JKM,LNG Futures,5/1/2026,JKM,C,11.5,7.914,-0.119,2026-04-15,4172,40.9523,1.0,19.414,26,0.071233
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,JKM,LNG Futures,6/1/2033,JKM,F,NaN,9.176,0.193,2033-05-13,4172,NaN,NaN,9.176,2611,7.153425
596,JKM,LNG Futures,7/1/2033,JKM,F,NaN,9.084,0.193,2033-06-15,4172,NaN,NaN,9.084,2644,7.243836
597,JKM,LNG Futures,8/1/2033,JKM,F,NaN,9.256,0.194,2033-07-15,4172,NaN,NaN,9.256,2674,7.326027
598,JKM,LNG Futures,9/1/2033,JKM,F,NaN,9.407,0.194,2033-08-15,4172,NaN,NaN,9.407,2705,7.410959


In [11]:
def BS_d1(S, K, r, sigma, t):
    d1 = (np.log(S/K) + (r+sigma*sigma*0.5)*t) / sigma*(np.sqrt(t))
    return d1

def BS_d2(BS_d1, sigma, t):
    d2 = BS_d1 - sigma*np.sqrt(t)
    return d2

def Black76_d1(F, K, sigma, t):
    d1 = (np.log(F/K) + (sigma*sigma*0.5)*t) / sigma*(np.sqrt(t))
    return d1

def Black76_d2(BS_d1, sigma, t):
    d2 = BS_d1 - sigma*np.sqrt(t)
    return d2

1. Calculate (Black Scholes, Black76 d1, d2)

In [12]:
F = df["SETTLEMENT PRICE_y"]
c_type = df["CONTRACT TYPE"]

K = df["STRIKE"]
r = 0.0369
sigma = df["OPTION_VOLATILITY"]*0.01
t = df["TIME TO MATURITY-365"]

df["BS_d1"] = BS_d1(F, K, r, sigma, t)
df["BS_d2"] = BS_d2(df["BS_d1"], sigma, t)
df["Black76_d1"] = Black76_d1(F, K, sigma, t)
df["Black76_d2"] = Black76_d2(df["Black76_d1"], sigma, t)

In [13]:
df

,HUB,PRODUCT,STRIP,CONTRACT,CONTRACT TYPE,STRIKE,SETTLEMENT PRICE_x,NET CHANGE,EXPIRATION DATE,PRODUCT_ID,OPTION_VOLATILITY,DELTA_FACTOR,SETTLEMENT PRICE_y,TIME TO MATURITY,TIME TO MATURITY-365,BS_d1,BS_d2,Black76_d1,Black76_d2
0,JKM,LNG Futures,5/1/2026,JKM,C,5.0,14.414,-0.119,2026-04-15,4172,103.7693,1.0,19.414,26,0.071233,0.358773,0.081818,0.358771,0.081816
1,JKM,LNG Futures,5/1/2026,JKM,C,8.0,11.414,-0.119,2026-04-15,4172,68.3377,1.0,19.414,26,0.071233,0.352745,0.170355,0.352742,0.170352
2,JKM,LNG Futures,5/1/2026,JKM,C,9.0,10.414,-0.119,2026-04-15,4172,59.4640,1.0,19.414,26,0.071233,0.350706,0.192000,0.350703,0.191997
3,JKM,LNG Futures,5/1/2026,JKM,C,10.0,9.414,-0.119,2026-04-15,4172,51.5178,1.0,19.414,26,0.071233,0.348589,0.211091,0.348585,0.211087
4,JKM,LNG Futures,5/1/2026,JKM,C,11.5,7.914,-0.119,2026-04-15,4172,40.9523,1.0,19.414,26,0.071233,0.345170,0.235870,0.345165,0.235865
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,JKM,LNG Futures,6/1/2033,JKM,F,NaN,9.176,0.193,2033-05-13,4172,NaN,NaN,9.176,2611,7.153425,NaN,NaN,NaN,NaN
596,JKM,LNG Futures,7/1/2033,JKM,F,NaN,9.084,0.193,2033-06-15,4172,NaN,NaN,9.084,2644,7.243836,NaN,NaN,NaN,NaN
597,JKM,LNG Futures,8/1/2033,JKM,F,NaN,9.256,0.194,2033-07-15,4172,NaN,NaN,9.256,2674,7.326027,NaN,NaN,NaN,NaN
598,JKM,LNG Futures,9/1/2033,JKM,F,NaN,9.407,0.194,2033-08-15,4172,NaN,NaN,9.407,2705,7.410959,NaN,NaN,NaN,NaN


### [Matrix] Call/Put Price - Time to Maturity x Strike

In [14]:
df[df["CONTRACT TYPE"] == "C"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='SETTLEMENT PRICE_x').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,14.414,13.861,13.509,13.094,12.811,12.287,12.235,12.398,12.307,...,4.122,4.138,4.199,4.375,4.535,4.657,4.772,4.817,4.764,3.901
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.341,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.405,...,2.630,2.658,2.725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.370,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,11.414,10.861,10.528,10.161,9.948,NaN,NaN,9.553,9.510,...,2.058,2.090,2.155,2.476,2.619,2.733,NaN,NaN,NaN,NaN
5,8.5,NaN,10.361,10.040,NaN,NaN,8.897,8.900,NaN,9.080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,10.414,9.861,9.558,9.231,9.063,8.443,8.459,8.680,8.663,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,9.081,NaN,NaN,7.999,8.031,8.263,8.259,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,8.472,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,8.390,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df[df["CONTRACT TYPE"] == "P"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='SETTLEMENT PRICE_x').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,0.001,0.001,0.001,0.002,0.009,0.003,0.006,0.011,0.019,...,0.144,0.154,0.160,0.271,0.271,0.275,0.279,0.290,0.308,0.423
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.053,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.117,...,0.652,0.674,0.686,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.931,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,0.001,0.001,0.019,0.069,0.146,NaN,NaN,0.166,0.222,...,1.080,1.106,1.116,1.372,1.355,1.351,NaN,NaN,NaN,NaN
5,8.5,NaN,0.001,0.031,NaN,NaN,0.113,0.171,NaN,0.292,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,0.001,0.001,0.049,0.139,0.261,0.159,0.230,0.293,0.375,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,0.072,NaN,NaN,0.215,0.302,0.376,0.471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,0.370,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,0.388,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### [Matrix] Call/Put Volatility [ICE] - Time to Maturity x Strike

In [16]:
df[df["CONTRACT TYPE"] == "C"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='OPTION_VOLATILITY').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,103.7693,71.3062,75.2076,77.6164,79.2320,61.6139,62.2649,62.7725,62.8751,...,30.5424,30.5961,30.5730,35.5293,35.6094,35.5766,35.6111,35.6163,35.5824,35.5842
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.9248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.9745,...,30.3018,30.3558,30.3706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,30.3011,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,68.3377,46.4409,75.2132,77.6221,79.2379,NaN,NaN,62.9212,63.0242,...,30.2066,30.2605,30.2557,35.1958,35.2754,35.2431,NaN,NaN,NaN,NaN
5,8.5,NaN,43.2351,75.2141,NaN,NaN,61.7857,62.4377,NaN,63.0490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,59.4640,43.0674,75.2147,77.6242,79.2401,61.8103,62.4624,62.9708,63.0739,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,75.2159,NaN,NaN,61.8348,62.4872,62.9956,63.0987,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,79.2415,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,79.2417,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### [Matrix] Call/Put Delta Factor[ICE] - Time to Maturity x Strike

In [17]:
df[df["CONTRACT TYPE"] == "C"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='DELTA_FACTOR').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,1.00000,1.00000,0.99989,0.99907,0.99706,0.99892,0.99778,0.99652,0.99467,...,0.93408,0.93170,0.93103,0.91108,0.91379,0.91520,0.91633,0.91540,0.91180,0.87802
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.98668,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.97334,...,0.78078,0.77944,0.78158,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.72696,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,1.00000,1.00000,0.99251,0.98011,0.96627,NaN,NaN,0.96196,0.95413,...,0.68478,0.68513,0.68962,0.69517,0.70616,0.71384,NaN,NaN,NaN,NaN
5,8.5,NaN,1.00000,0.98841,NaN,NaN,0.96747,0.95753,NaN,0.94233,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,1.00000,0.99999,0.98288,0.96338,0.94484,0.95667,0.94544,0.93856,0.92915,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,0.97572,NaN,NaN,0.94397,0.93165,0.92460,0.91466,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,0.92637,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,0.92351,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df[df["CONTRACT TYPE"] == "P"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='DELTA_FACTOR').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,0.00000,0.00000,-0.00011,-0.00093,-0.00294,-0.00108,-0.00223,-0.00348,-0.00534,...,-0.06592,-0.06830,-0.06897,-0.08892,-0.08621,-0.08480,-0.08367,-0.08460,-0.08820,-0.12198
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.01332,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.02666,...,-0.21922,-0.22056,-0.21842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-0.27304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,0.00000,0.00000,-0.00749,-0.01989,-0.03373,NaN,NaN,-0.03804,-0.04587,...,-0.31522,-0.31487,-0.31038,-0.30483,-0.29384,-0.28616,NaN,NaN,NaN,NaN
5,8.5,NaN,0.00000,-0.01159,NaN,NaN,-0.03253,-0.04247,NaN,-0.05767,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,0.00000,-0.00001,-0.01712,-0.03662,-0.05516,-0.04333,-0.05456,-0.06144,-0.07086,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,-0.02428,NaN,NaN,-0.05603,-0.06836,-0.07540,-0.08534,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,-0.07363,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,-0.07649,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df[df["CONTRACT TYPE"] == "P"].pivot_table(index='STRIKE',columns='TIME TO MATURITY', values='OPTION_VOLATILITY').reset_index()

TIME TO MATURITY,STRIKE,26,56,87,117,147,179,209,238,270,...,818,847,879,910,938,971,1001,1032,1063,1091
0,5.0,103.7693,71.3062,75.2076,77.6164,79.2320,61.6139,62.2649,62.7725,62.8751,...,30.5424,30.5961,30.5730,35.5293,35.6094,35.5766,35.6111,35.6163,35.5824,35.5842
1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.9248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.9745,...,30.3018,30.3558,30.3706,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,30.3011,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8.0,68.3377,46.4409,75.2132,77.6221,79.2379,NaN,NaN,62.9212,63.0242,...,30.2066,30.2605,30.2557,35.1958,35.2754,35.2431,NaN,NaN,NaN,NaN
5,8.5,NaN,43.2351,75.2141,NaN,NaN,61.7857,62.4377,NaN,63.0490,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,9.0,59.4640,43.0674,75.2147,77.6242,79.2401,61.8103,62.4624,62.9708,63.0739,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,9.5,NaN,NaN,75.2159,NaN,NaN,61.8348,62.4872,62.9956,63.0987,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9.7,NaN,NaN,NaN,NaN,79.2415,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9.8,NaN,NaN,NaN,NaN,79.2417,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2. Calculate Delta, Gamma, Vega, Rho (Based on Option Volatility Given (ICE))

In [20]:
def Black76_delta(c_type, F, K, r, sigma, t, d1):
    
    delta = np.where(
        c_type == 'C',
        np.exp(-r * t) * norm.cdf(d1),       
        -np.exp(-r * t) * norm.cdf(-d1)
    )
    
    return delta

def Black76_gamma(F, K, r, sigma, t, d1):
    gamma = np.exp(-r*t) * norm.pdf(d1, loc=0, scale=1) / F * sigma * np.sqrt(t)
    return gamma

def Black76_vega(S, K, r, sigma, t,d1):
    gamma = F * np.exp(-r*t) *  norm.pdf(d1, loc=0, scale=1) * np.sqrt(t)
    return gamma

# def Black76_rho(S, K, r, sigma, t):
#     gamma = F * np.exp(-r*t) *  norm.pdf(d1, loc=0, scale=1) * np.sqrt(t)
#     return gamma

In [21]:
df["Black76_delta"] = Black76_delta(c_type, F, K, r, sigma, t, df["Black76_d1"])

In [22]:
df["Black76_gamma"] = Black76_gamma(F, K, r, sigma, t, df["Black76_d1"])

In [23]:
df["Black76_vega"] = Black76_vega(F, K, r, sigma, t, df["Black76_d1"])

In [24]:
df

,HUB,PRODUCT,STRIP,CONTRACT,CONTRACT TYPE,STRIKE,SETTLEMENT PRICE_x,NET CHANGE,EXPIRATION DATE,PRODUCT_ID,...,SETTLEMENT PRICE_y,TIME TO MATURITY,TIME TO MATURITY-365,BS_d1,BS_d2,Black76_d1,Black76_d2,Black76_delta,Black76_gamma,Black76_vega
0,JKM,LNG Futures,5/1/2026,JKM,C,5.0,14.414,-0.119,2026-04-15,4172,...,19.414,26,0.071233,0.358773,0.081818,0.358771,0.081816,0.640112,0.005336,1.938259
1,JKM,LNG Futures,5/1/2026,JKM,C,8.0,11.414,-0.119,2026-04-15,4172,...,19.414,26,0.071233,0.352745,0.170355,0.352742,0.170352,0.637854,0.003522,1.942421
2,JKM,LNG Futures,5/1/2026,JKM,C,9.0,10.414,-0.119,2026-04-15,4172,...,19.414,26,0.071233,0.350706,0.192000,0.350703,0.191997,0.637090,0.003067,1.943814
3,JKM,LNG Futures,5/1/2026,JKM,C,10.0,9.414,-0.119,2026-04-15,4172,...,19.414,26,0.071233,0.348589,0.211091,0.348585,0.211087,0.636295,0.002659,1.945254
4,JKM,LNG Futures,5/1/2026,JKM,C,11.5,7.914,-0.119,2026-04-15,4172,...,19.414,26,0.071233,0.345170,0.235870,0.345165,0.235865,0.635010,0.002116,1.947563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,JKM,LNG Futures,6/1/2033,JKM,F,NaN,9.176,0.193,2033-05-13,4172,...,9.176,2611,7.153425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
596,JKM,LNG Futures,7/1/2033,JKM,F,NaN,9.084,0.193,2033-06-15,4172,...,9.084,2644,7.243836,NaN,NaN,NaN,NaN,NaN,NaN,NaN
597,JKM,LNG Futures,8/1/2033,JKM,F,NaN,9.256,0.194,2033-07-15,4172,...,9.256,2674,7.326027,NaN,NaN,NaN,NaN,NaN,NaN,NaN
598,JKM,LNG Futures,9/1/2033,JKM,F,NaN,9.407,0.194,2033-08-15,4172,...,9.407,2705,7.410959,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- With a fixed interest rate, the Delta of Call option that are heavily ITM does not go to 1 (error in calculation, needs to have full interest rate curve)

3. Invert the Market Prices to obtain Implied Volatility for prices that are known

4. Plot Time to maturity, Strike, volatility as 3d 

5. Interpolating for interest rates (1m SOFR)

6. Using interest rates to back out Implied Vol (Not full spectrum of Implied Vol)

